In [1]:
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql.window import Window
from pyspark.sql.session import SparkSession
from IPython.display import display

In [2]:
spark = SparkSession.builder\
       .master("local[*]")\
       .appName("Spark_App")\
       .getOrCreate()

In [3]:
df = spark.read.options(header='True', inferSchema='True', delimiter=',') \
.csv('Vermont_Vendor_Payments.csv')

display(df.limit(5).toPandas())

,Quarter Ending,Department,UnitNo,Vendor Number,Vendor,City,State,DeptID Description,DeptID,Amount,Account,AcctNo,Fund Description,Fund
0,09/30/2009,Environmental Conservation,6140.0,0000276016,1st Run Computer Services Inc,None,NY,WQD - Waterbury,6.140040e+09,930.0,Rep&Maint-Info Tech Hardware,513000.0,Environmental Permit Fund,21295.0
1,09/30/2009,Environmental Conservation,6140.0,0000276016,1st Run Computer Services Inc,None,NY,Water Supply Division - Wtby,6.140040e+09,930.0,Rep&Maint-Info Tech Hardware,513000.0,Environmental Permit Fund,21295.0
2,09/30/2009,Vermont Veterans' Home,3300.0,0000284121,210 Innovations LLC,None,CT,MAINTENANCE,3.300010e+09,24.0,Freight & Express Mail,517300.0,Vermont Medicaid,21782.0
3,09/30/2009,Vermont Veterans' Home,3300.0,0000284121,210 Innovations LLC,None,CT,MAINTENANCE,3.300010e+09,420.0,Building Maintenance Supplies,520200.0,Vermont Medicaid,21782.0
4,09/30/2009,Corrections,3480.0,0000207719,21st Century Cellular,None,PA,Brattleboro P&P,3.480005e+09,270.8,Telecom-Wireless Phone Service,516659.0,General Fund,10000.0


In [4]:
df_new = df.groupBy("Department").agg( f.count(f.lit(1)).alias("Count")
                                      , (f.sum("Amount")/f.lit(1000000)).alias("Sum Amount (M)")
                                      , f.max("Amount").alias("Max Amount")
                                      , f.min("Amount").alias("Min Amount"))\
                                    .orderBy(f.col("Count").desc())

display(df_new.distinct().limit(10).toPandas())

,Department,Count,Sum Amount (M),Max Amount,Min Amount
0,Transportation Agency,163552,2891.035008,19572633.75,-926250.00
1,Children and Families,130847,1926.385405,15393655.22,-251657.66
2,Agency of Transportation,102395,1920.833560,24976543.73,-175077.00
3,Buildings & Gen Serv-Prop,98528,518.377059,4659131.24,-167783.00
4,Children and Family Services,94097,1281.368751,18606559.02,-149374.07
5,Corrections,76231,717.989833,7189419.02,-91910.00
6,Health,73193,853.538376,7992250.27,-219667.00
7,Judiciary,63311,151.577875,1392999.00,-15000.00
8,Public Safety,58697,344.163692,6732564.00,-230279.93
9,Education Agency,47785,5207.509929,9830559.14,-597550.82


In [5]:
df_new = df.select(f.upper("Department").alias('Department'))

display(df_new.distinct().limit(10).toPandas())

,Department
0,LIBRARIES
1,GOVERNOR'S COMMISSION ON WOMEN
2,AOT PROPRIETARY FUNDS
3,FINANCE & MANAGEMENT
4,VT HOUSING & CONSERV BOARD
5,SECRETARY OF STATE'S OFFICE
6,"FOREST, PARKS & RECREATION"
7,NATURAL RESOURCES AGENCY
8,AUDITOR OF ACCOUNTS-PROP
9,ATTORNEY GENERAL'S OFFICE


In [6]:
df_new = df.select(f.lower("Department").alias('Department'))

display(df_new.distinct().limit(10).toPandas())

,Department
0,sergeant at arms
1,veterans' home
2,transportation agency
3,office of the attorney general
4,green mountain care board
5,joint fiscal office
6,vt offender work program
7,office of the defender general
8,education agency
9,office of vt health access


In [7]:
df_new = df.select(f.initcap("Department").alias('Department'))

display(df_new.distinct().limit(10).toPandas())

,Department
0,Agency Of Digital Services
1,Labor
2,Buildings & Gen Serv-prop
3,Vermont Health Access
4,Agency Of Transportation
5,Public Service Board
6,Natural Resources Agency
7,Education
8,Vt Offender Work Program
9,Administration Agency


In [8]:
df_new = df.select(f.col('Account'), f.length('Account').alias('Account length'))

display(df_new.distinct().limit(10).toPandas())

,Account,Account length
0,IT Contracts - Application Sup,30
1,Rent Land&Bldgs-Non-Office,26
2,Misc Equipment Rental,21
3,Client Meetings-Econ Dev Only,29
4,Community Supports,18
5,Software - Application Support,30
6,DBVI Innovation & Expansion,27
7,VSNIP,5
8,Fuel,4
9,Success by Six,14


In [9]:
df_new = df.select(f.col('UnitNo'), f.lpad('UnitNo', 6, '0').alias('Padded UnitNo'))

display(df_new.distinct().limit(10).toPandas())

,UnitNo,Padded UnitNo
0,7110.0,7110.0
1,7150.0,7150.0
2,1255.0,1255.0
3,3150.0,3150.0
4,4100.0,4100.0
5,6130.0,6130.0
6,3410.0,3410.0
7,2100.0,2100.0
8,1120.0,1120.0
9,7100.0,7100.0


In [10]:
df_new = df.select(f.trim("Department").alias('Department'))

display(df_new.distinct().limit(10).toPandas())

,Department
0,Labor
1,Vermont Health Access
2,Public Service Board
3,Natural Resources Agency
4,Children and Family Services
5,Aging and Independent Living
6,Education
7,Administration Agency
8,Children and Families
9,DLL - Div of Liquor Control


In [11]:
df_new = df.select(f.concat(f.col("UnitNo"), f.lit(" "), f.col("Department")).alias('UnitNo Department'))

display(df_new.distinct().limit(10).toPandas())

,UnitNo Department
0,2160.0 Center of Crime Victims' Serv
1,1115.0 F&M - Financial Management Sys
2,1125.0 Personnel-Proprietary
3,2110.0 Office of the Defender General
4,6100.0 Natural Res Central Office
5,8110.0 AOT Proprietary Funds
6,1180.0 Buildings & Gen Serv-Capital
7,7150.0 Vermont Life
8,6140.0 Environmental Conservation
9,3300.0 Vermont Veterans' Home


In [12]:
df_new = df.select(f.concat_ws(" ",f.col("UnitNo"), f.col("Department")).alias('UnitNo Department'))

display(df_new.distinct().limit(10).toPandas())

,UnitNo Department
0,2160.0 Center of Crime Victims' Serv
1,1115.0 F&M - Financial Management Sys
2,1125.0 Personnel-Proprietary
3,2110.0 Office of the Defender General
4,6100.0 Natural Res Central Office
5,8110.0 AOT Proprietary Funds
6,1180.0 Buildings & Gen Serv-Capital
7,7150.0 Vermont Life
8,6140.0 Environmental Conservation
9,3300.0 Vermont Veterans' Home


In [13]:
df_new = df.filter(f.col('Department').contains('Office'))\
.select(f.concat_ws(" ",f.col("UnitNo"), f.col("Department")).alias('UnitNo Department'))

display(df_new.distinct().limit(10).toPandas())

,UnitNo Department
0,2110.0 Office of the Defender General
1,6100.0 Natural Res Central Office
2,2100.0 Office of the Attorney General
3,2100.0 Attorney General's Office
4,1100.0 Agency of Admin Sec Office
5,1220.0 Joint Fiscal Office
6,2230.0 Secretary of State's Office
7,1260.0 Treasurer's Office
8,2110.0 Defender General's Office
9,1250.0 Auditor of Accounts' Office


In [14]:
df_new = df.filter(f.col('Vendor').startswith('1'))\
.select(f.concat_ws(", ",f.col("Vendor"), f.col("State")).alias('Vendor State'))

display(df_new.distinct().limit(10).toPandas())

,Vendor State
0,"1 South Main Supply, VT"
1,"116 Main Street LLC, VT"
2,"1st Attack Engineering Inc, IN"
3,"12 Gauge Electric LLC, VT"
4,"1 Source International LLC, GA"
5,"1st Run Computer Services Inc, NY"
6,"119 Main Street Investors &, VT"
7,"1016 Route 5 LLC, VT"
8,"1000 Stone Farm, LLC, VT"
9,"1mage Software Inc, CO"


In [15]:
df_new = df.filter(f.col('Vendor').endswith('LLC'))\
.select(f.concat_ws(", ",f.col("Vendor"), f.col("State")).alias('Vendor State'))

display(df_new.distinct().limit(10).toPandas())

,Vendor State
0,"Bjurling Law, PLLC, VT"
1,"CleanEdison LLC, NY"
2,"DocSite LLC, MD"
3,"VARIDESK, LLC, TX"
4,"JoLu LLC, VT"
5,"DEW Prospect Street LLC, VT"
6,"DLP Hospitality LLC, MA"
7,"Housing PV 1 LLC, VT"
8,"Vermont HydroGeo, LLC, VT"
9,"Top Trowel Masonry, LLC, VT"


In [16]:
df_new = df.select(f.split("Department", " ").alias('Department list'))\
.select("*", f.concat_ws("-",f.col("Department list")).alias('Department concat'))

display(df_new.distinct().limit(10).toPandas())

,Department list,Department concat
0,"[Buildings, &, Gen, Serv-Gov'tal]",Buildings-&-Gen-Serv-Gov'tal
1,"[Agency, of, Digital, Services]",Agency-of-Digital-Services
2,[Retirement],Retirement
3,[Military],Military
4,"[Agency, of, Admin, Sec, Office]",Agency-of-Admin-Sec-Office
5,"[Liquor, Control]",Liquor-Control
6,[Education],Education
7,"[State, Treasurer-Gov'tal]",State-Treasurer-Gov'tal
8,"[Agriculture,, Food&Mrkts, Agency]","Agriculture,-Food&Mrkts-Agency"
9,[Personnel-Governmental],Personnel-Governmental


In [17]:
df_new = df.select(f.col('UnitNo'), f.translate('UnitNo', '136', 'XYZ').alias('Translated UnitNo'))

display(df_new.distinct().limit(10).toPandas())

,UnitNo,Translated UnitNo
0,1230.0,X2Y0.0
1,2130.0,2XY0.0
2,3410.0,Y4X0.0
3,3300.0,YY00.0
4,3480.0,Y480.0
5,2140.0,2X40.0
6,1280.0,X280.0
7,6100.0,ZX00.0
8,2230.0,22Y0.0
9,3675.0,YZ75.0


In [18]:
df_new = df.selectExpr("State", "case when State = 'NY' or State = 'BC' then 1 else 0 end as Is_NY_BC")

display(df_new.distinct().orderBy(f.col("Is_NY_BC").desc()).limit(10).toPandas())

,State,Is_NY_BC
0,NY,1
1,BC,1
2,IN,0
3,ZZ,0
4,MD,0
5,VA,0
6,AZ,0
7,PE,0
8,GE,0
9,OK,0


In [19]:
window = Window.partitionBy("Account").orderBy(f.col("Amount").desc())

df_new = df.withColumn("row", f.row_number().over(window)) \
  .filter(f.col("row") == 1).drop("row") \
  .select("Amount",	"Account", "AcctNo", "Fund Description", "Fund")

display(df_new.distinct().limit(10).toPandas())

,Amount,Account,AcctNo,Fund Description,Fund
0,702013.00,AAA Area Plan Programs,608580.0,Federal Revenue Fund,22005.0
1,641712.00,AAA Grants,605070.0,Global Commitment Fund,20405.0
2,3477000.00,AABD,604200.0,General Fund,10000.0
3,98248.50,ADAP Prevention Activities,602910.0,Global Commitment Fund,20405.0
4,85500.00,ADAP Recovery Centers,602920.0,Global Commitment Fund,20405.0
5,231.21,ADL Items/Self-help Aids,601093.0,Federal Revenue Fund,22005.0
6,8365.00,ADR Mediation,507505.0,General Fund,10000.0
7,151215.87,ADS ACD Exp,516686.0,Tax-Miscellaneous Fees,21590.0
8,1663809.86,ADS Allocation Exp,516685.0,General Fund,10000.0
9,1790048.54,ADS App Support SOV Emp Exp,516661.0,Transp Fund - Nondedicated,20105.0


In [20]:
df_new = df.select("Department").distinct().filter("Department is not null")

window = Window.orderBy(f.col("Department").asc())

df_new = df_new.withColumn("Row number", f.row_number().over(window))

display(df_new.distinct().limit(10).toPandas())

,Department,Row number
0,AOT Proprietary Funds,1
1,Administration Agency,2
2,Agency of Admin Sec Office,3
3,Agency of Digital Services,4
4,Agency of Transportation,5
5,Aging & Ind Living-Proprietary,6
6,Aging and Independent Living,7
7,"Agriculture, Food & Markets",8
8,"Agriculture, Food&Mrkts Agency",9
9,Attorney General's Office,10


In [21]:
df_new = df.select("Amount").summary()

display(df_new.distinct().limit(10).toPandas())

,summary,Amount
0,count,1680170
1,mean,29244.06047890568
2,stddev,1526395.088975898
3,min,-2880183.34
4,25%,68.48
5,50%,371.4
6,75%,2250.41
7,max,1.50113710631E9


In [22]:
window = Window.orderBy(f.col("Amount").asc())

df_new = df.select("Amount").withColumn("Amount_tile", f.ntile(10).over(window))

df_new = df_new.groupBy("Amount_tile").agg(f.min("Amount").alias("Min Amount")
                                           , f.max("Amount").alias("Max Amount"))

display(df_new.limit(10).toPandas())

,Amount_tile,Min Amount,Max Amount
0,1,-2880183.34,1.800000e+01
1,2,18.00,4.689000e+01
2,3,46.89,1.000000e+02
3,4,100.00,1.939600e+02
4,5,193.96,3.714600e+02
5,6,371.47,7.105700e+02
6,7,710.57,1.485440e+03
7,8,1485.44,3.708460e+03
8,9,3708.48,1.462983e+04
9,10,14630.00,1.501137e+09


In [23]:
data_dict = [
    {"date": "2022-01", "item": "apple", "sales": 100},
    {"date": "2022-01", "item": "banana", "sales": 200},
    {"date": "2022-01", "item": "orange", "sales": 300},
    {"date": "2022-02", "item": "apple", "sales": 150},
    {"date": "2022-02", "item": "banana", "sales": 250},
    {"date": "2022-02", "item": "orange", "sales": 350},
    {"date": "2022-03", "item": "apple", "sales": 200},
    {"date": "2022-03", "item": "banana", "sales": 300},
    {"date": "2022-03", "item": "orange", "sales": 400},
    {"date": "2022-04", "item": "apple", "sales": 700},
    {"date": "2022-04", "item": "banana", "sales": 200},
    {"date": "2022-04", "item": "orange", "sales": 500},
    {"date": "2022-05", "item": "apple", "sales": 700},
    {"date": "2022-05", "item": "banana", "sales": 200},
    {"date": "2022-05", "item": "orange", "sales": 500}
]

sales_data = spark.createDataFrame(data_dict)
sales_data = sales_data.select("item", "date", "sales")

window = Window.partitionBy("item").orderBy("date")

sales_data = sales_data.withColumn("prev_sales", f.lag("sales", 1).over(window))
sales_data = sales_data.withColumn("next_sales", f.lead("sales", 1).over(window))
sales_data = sales_data.withColumn("sales_pct_change", (f.col("sales") - f.col("prev_sales")) / f.col("prev_sales"))

display(sales_data.toPandas())

,item,date,sales,prev_sales,next_sales,sales_pct_change
0,apple,2022-01,100,NaN,150.0,NaN
1,apple,2022-02,150,100.0,200.0,0.500000
2,apple,2022-03,200,150.0,700.0,0.333333
3,apple,2022-04,700,200.0,700.0,2.500000
4,apple,2022-05,700,700.0,NaN,0.000000
5,banana,2022-01,200,NaN,250.0,NaN
6,banana,2022-02,250,200.0,300.0,0.250000
7,banana,2022-03,300,250.0,200.0,0.200000
8,banana,2022-04,200,300.0,200.0,-0.333333
9,banana,2022-05,200,200.0,NaN,0.000000


In [24]:
data_dict = [
    {"date": "2022-01", "item": "apple", "sales": 100},
    {"date": "2022-01", "item": "banana", "sales": 200},
    {"date": "2022-01", "item": "orange", "sales": 300},
    {"date": "2022-02", "item": "apple", "sales": 150},
    {"date": "2022-02", "item": "banana", "sales": 250},
    {"date": "2022-02", "item": "orange", "sales": 350},
    {"date": "2022-03", "item": "apple", "sales": 200},
    {"date": "2022-03", "item": "banana", "sales": 300},
    {"date": "2022-03", "item": "orange", "sales": 400},
    {"date": "2022-04", "item": "apple", "sales": 700},
    {"date": "2022-04", "item": "banana", "sales": 200},
    {"date": "2022-04", "item": "orange", "sales": 500},
    {"date": "2022-05", "item": "apple", "sales": 700},
    {"date": "2022-05", "item": "banana", "sales": 200},
    {"date": "2022-05", "item": "orange", "sales": 500}
]

sales_data = spark.createDataFrame(data_dict).selectExpr("item"
                                                         , "date"
                                                         , "case when date like '%02' or date like '%04' then null else sales end as sales")

window = Window.partitionBy("item").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_data = sales_data.withColumn("Cumulative sales", f.sum("Sales").over(window))
sales_data = sales_data.withColumn("Running average", f.avg("Sales").over(window))
sales_data = sales_data.withColumn("First", f.first("Sales").over(window))
sales_data = sales_data.withColumn("Last", f.last("Sales").over(window))

window = Window.partitionBy("item").orderBy("date").rowsBetween(Window.currentRow, Window.unboundedFollowing)

sales_data = sales_data.withColumn("Future sales", f.sum("Sales").over(window))

window = Window.partitionBy("item").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

sales_data = sales_data.withColumn("Total sales", f.sum("Sales").over(window))

sales_data = sales_data.withColumn("Percent of total", f.col("Cumulative sales")/f.col("Total sales"))

display(sales_data.toPandas())

,item,date,sales,Cumulative sales,Running average,First,Last,Future sales,Total sales,Percent of total
0,apple,2022-01,100.0,100,100.000000,100,100.0,1000,1000,0.100000
1,apple,2022-02,NaN,100,100.000000,100,NaN,900,1000,0.100000
2,apple,2022-03,200.0,300,150.000000,100,200.0,900,1000,0.300000
3,apple,2022-04,NaN,300,150.000000,100,NaN,700,1000,0.300000
4,apple,2022-05,700.0,1000,333.333333,100,700.0,700,1000,1.000000
5,banana,2022-01,200.0,200,200.000000,200,200.0,700,700,0.285714
6,banana,2022-02,NaN,200,200.000000,200,NaN,500,700,0.285714
7,banana,2022-03,300.0,500,250.000000,200,300.0,500,700,0.714286
8,banana,2022-04,NaN,500,250.000000,200,NaN,200,700,0.714286
9,banana,2022-05,200.0,700,233.333333,200,200.0,200,700,1.000000


In [25]:
data_dict = [
    {"date": "2022-01", "item": "apple", "status": "A"},
    {"date": "2022-01", "item": "banana", "status": "B"},
    {"date": "2022-01", "item": "orange", "status": "C"},
    {"date": "2022-02", "item": "apple", "status": "AA"},
    {"date": "2022-02", "item": "banana", "status": "BB"},
    {"date": "2022-02", "item": "orange", "status": "CC"},
    {"date": "2022-03", "item": "apple", "status": "B"},
    {"date": "2022-03", "item": "banana", "status": "C"},
    {"date": "2022-03", "item": "orange", "status": "D"},
    {"date": "2022-04", "item": "apple", "status": "H"},
    {"date": "2022-04", "item": "banana", "status": "B"},
    {"date": "2022-04", "item": "orange", "status": "F"},
    {"date": "2022-05", "item": "apple", "status": "H"},
    {"date": "2022-05", "item": "banana", "status": "B"},
    {"date": "2022-05", "item": "orange", "status": "F"}
]

status_data = spark.createDataFrame(data_dict).selectExpr("item"
                                                         , "date"
                                                         , "case when date like '%02' or date like '%04' then null else status end as status")

window = Window.partitionBy("item").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

status_data = status_data.withColumn("status_list", f.collect_list("status").over(window))

status_data = status_data.withColumn("last_status", f.element_at("status_list", -1))

status_data = status_data.withColumn("status_list_cum", f.collect_list("last_status").over(window))
status_data = status_data.withColumn("cumulative_status", f.array_join("status_list_cum", ""))

display(status_data.toPandas())

,item,date,status,status_list,last_status,status_list_cum,cumulative_status
0,apple,2022-01,A,[A],A,[A],A
1,apple,2022-02,None,[A],A,"[A, A]",AA
2,apple,2022-03,B,"[A, B]",B,"[A, A, B]",AAB
3,apple,2022-04,None,"[A, B]",B,"[A, A, B, B]",AABB
4,apple,2022-05,H,"[A, B, H]",H,"[A, A, B, B, H]",AABBH
5,banana,2022-01,B,[B],B,[B],B
6,banana,2022-02,None,[B],B,"[B, B]",BB
7,banana,2022-03,C,"[B, C]",C,"[B, B, C]",BBC
8,banana,2022-04,None,"[B, C]",C,"[B, B, C, C]",BBCC
9,banana,2022-05,B,"[B, C, B]",B,"[B, B, C, C, B]",BBCCB
